In [ ]:
!wget https://phonetics-spbu.github.io/courses/python_textbook/files/segtools.py

1. Написать функцию, которая принимает на вход последовательность транскрипционных знаков CORPRES (список), а возвращает список контекстных аллофонов (трифонов), каждый из которых - кортеж вида:

("левый контекст", "звук", "правый контекст")

Обозначения CORPRES: ш - sh, ж - zh, щ - sc, ц - c, ч - ch, х - h, ы - y

Звонкие аллофоны глухих: C, H, CH, SC. Твёрдый аллофон: ch_

Например: j u0 r' i4 j -> `[("#", "j", "rounded vowel"), ("j", "u0", "palatalized"), ("vowel", "r'", "unrounded vowel"), ...]`

Рекомендуемая архитектура:
1. завести 4 словаря соответствий: левые контексты согласных, правые контексты согласных, …
2. в каждом словаре ключи - названия групп, значения - списки звуков из этой группы. Список звуков легко задать так:
`vowels = "i0 e0 a0 o0 u0 y0".split()`
3. сделать обратные словари, где ключи - звуки, значения - названия групп. Для этого нужно пройти циклом по изначальным словарям и пройти циклом по каждому списку из словаря
использовать обратные словари для генерации кортежей


In [ ]:
import re
from collections import defaultdict

import numpy as np
import scipy.io.wavfile as wavfile

import segtools

In [2]:
vowels = "i e a o u y".split()

consonants = (
    "p b t d k g c ch C ch_ "
    "p' b' t' d' k' g' CH "
    "f v s z sh zh h H"
    "f' v' s' z' sc h'"
    "m n l r "
    "m' n' l' r' j"
).split()

In [3]:
class2sound_vl = {
    "labials": "f v p b".split(),
    "labials_palat": "f' v' p' b'".split(),
    "dentals": "t d s z c".split(),
    "dentals_palat": "t' d' s' z' ch sc r' zh'".split(),
    "postalveolars": "sh zh ch_".split(),
    "velars": "k g h".split(),
    "velars_palat": "k' g' h'".split(),
    "start": [""]
}
class2sound_vl.update({i: [i] for i in "m m' n n' l l' r j a o u e i y".split()})

In [4]:
sound2class_vl = {j: i for i in class2sound_vl for j in class2sound_vl[i]}

In [5]:
class2sound_vr = {
    "labials": "p b m f v".split(),
    "dentals": "t d s z c C n r".split(),
    "postalveolars": "sh zh ch_".split(),
    "velars": "k g h H".split(),
    "palat": "f' v' p' b' t' d' s' z' ch sc r' zh' k' g' h' m' n' l' r'".split(),
    "end": [""]
}
class2sound_vr.update({i: [i] for i in "l j a o u e i y".split()})

In [6]:
sound2class_vr = {j: i for i in class2sound_vr for j in class2sound_vr[i]}

In [7]:
class2sound_cl = {
    "vowels": vowels,
    "consonants": consonants,
    "start": [""]
}

In [8]:
sound2class_cl = {j: i for i in class2sound_cl for j in class2sound_cl[i]}

In [9]:
class2sound_cr = {
    "rounded": "o u".split(),
    "unrounded": "i e a y".split(),
    "voiceless": "t k s p s' t' sh ch f h c k' sc p' f' h' ch_".split(),
    "sonorants": "n l v r n' m l' r' j m' v'".split(),
    "voiced": "d z g b zh d' b' z' g' H CH C".split(),
    "end": [""]
}

In [10]:
sound2class_cr = {j: i for i in class2sound_cr for j in class2sound_cr[i]}

In [15]:
def add_context(sounds):
    sounds = [""] + sounds + [""]
    result = []
    for s1, s2, s3 in zip(sounds, sounds[1:], sounds[2:]):
        if s2.startswith(tuple(vowels)):
            cur_class = "vowel"
        else:
            cur_class = "consonant"
        s1 = s1.rstrip("0124")
        s3 = s3.rstrip("0124")

        sound2class_left = sound2class_cl if cur_class == "consonant" else sound2class_vl
        sound2class_right = sound2class_cr if cur_class == "consonant" else sound2class_vr

        left = sound2class_left.get(s1, "UNK")
        right = sound2class_right.get(s3, "UNK")
        
        result.append((left, s2, right))
    return result

In [16]:
add_context("j u0 r' i t r' i0 f a4 n a4 f".split())

[('start', 'j', 'rounded'),
 ('j', 'u0', 'palat'),
 ('vowels', "r'", 'unrounded'),
 ('dentals_palat', 'i', 'dentals'),
 ('vowels', 't', 'sonorants'),
 ('consonants', "r'", 'unrounded'),
 ('dentals_palat', 'i0', 'labials'),
 ('vowels', 'f', 'unrounded'),
 ('labials', 'a4', 'dentals'),
 ('vowels', 'n', 'unrounded'),
 ('n', 'a4', 'labials'),
 ('vowels', 'f', 'end')]

2. Составить базу данных аллофонов по файлам SBL и SEG_B1. Оформить в виде словаря: ключи &mdash; контекстные аллофоны, значения &mdash; списки словарей: имя файла, начало и конец (в отсчётах). Списки отсортировать так, чтобы сначала шли те реализации, длительность которых ближе всего к средней.

3. Написать функцию, которая принимает на вход последовательность аллофонов и компилирует звуковую последовательность